# EVA — Обучение в DataSphere
233M параметров, 50000 шагов. Веса → /tmp/, снапшоты → диск.


In [ ]:
# 1. Установка
import os, sys

if not os.path.exists('/home/jupyter/EVA'):
    !git clone https://github.com/BlackCatSpb/FCF.git /home/jupyter/EVA
%cd /home/jupyter/EVA
!git pull 2>/dev/null; true

# Только недостающие пакеты, PyTorch НЕ трогаем
!pip install loguru tokenizers numpy faiss-cpu datasets huggingface_hub -q

!rm -rf /home/jupyter/EVA/snapshots
!rm -rf /home/jupyter/EVA/checkpoints
!mkdir -p /home/jupyter/EVA/snapshots
!mkdir -p /home/jupyter/EVA/logs

print("\n✅ Готово")
print(f"PyTorch: {sys.modules.get('torch', 'не загружен')}")
!df -h /tmp

In [ ]:
# 2. Проверка GPU и датасета
import torch, os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("GPU не найден")

ds = '/home/jupyter/EVA/real_data/combined_ru.txt'
if os.path.exists(ds):
    print(f"✅ Датасет: {os.path.getsize(ds) / 1024 / 1024:.1f} MB")
else:
    print("❌ combined_ru.txt не найден! Загрузите в /home/jupyter/EVA/real_data/")

In [ ]:
# 3. Обучение
import sys
sys.path.insert(0, '/home/jupyter/EVA')

from eva.config import FCFConfig
from eva.primordial_layer import PrimordialLayer
from eva.tokenizer_utils import load_tokenizer
from eva.language_trainer import LanguageTrainer
from eva.unified_grammar import UnifiedStateGrammar
import torch, os, json, time, pickle, faiss, gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Устройство: {device}")

config = FCFConfig()
config.training.learning_rate = 1e-4
config.training.max_steps = 50000

layer = PrimordialLayer(config)
if device == 'cuda':
    layer = layer.cuda()
print(f"Создан: {layer.summary()}")

tokenizer = load_tokenizer('/home/jupyter/EVA/tokenizer.json')
grammar = UnifiedStateGrammar(config.d_model)

trainer = LanguageTrainer(
    layer=layer, tokenizer=tokenizer, config=config,
    checkpoint_dir='/home/jupyter/EVA/checkpoints',
    state_grammar=grammar, benchmark_interval=2000,
)
trainer.checkpoint_interval = 2000
trainer.gen_test_interval = 2000

train_file = '/home/jupyter/EVA/real_data/combined_ru.txt'
snapshot_dir = '/home/jupyter/EVA/snapshots'
os.makedirs(snapshot_dir, exist_ok=True)

def save(final=False):
    p = os.path.join(snapshot_dir, f"step_{trainer.step:06d}" if not final else "final")
    os.makedirs(p, exist_ok=True)
    with open(os.path.join(p, 'snapshots.pkl'), 'wb') as f:
        pickle.dump(trainer.layer.state_storage.snapshots_meta, f)
    if trainer.layer.state_storage.index is not None:
        faiss.write_index(trainer.layer.state_storage.index, os.path.join(p, 'index.faiss'))
    with open(os.path.join(p, 'meta.pkl'), 'wb') as f:
        pickle.dump({'usage_count': trainer.layer.meta.usage_count, 'confidence_history': trainer.layer.meta.confidence_history, 'created_at': trainer.layer.meta.created_at}, f)
    trainer.layer.config.to_json(os.path.join(p, 'config.json'))
    torch.save(trainer.layer.state_dict(), '/tmp/latest_weights.pt')
    s = {'step': trainer.step, 'snapshots': len(trainer.layer.state_storage.snapshots_meta), 'confidence': trainer.layer.meta.average_confidence(), 'timestamp': time.time()}
    with open(os.path.join(p, 'status.json'), 'w') as f:
        json.dump(s, f)
    print(f"[Save] step={trainer.step} snap={s['snapshots']} conf={s['confidence']:.3f}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

trainer._save_checkpoint = save

print("\n" + "="*60)
print("  EVA — Обучение")
print("="*60)
print(f"  Устройство: {device}")
print(f"  Веса: /tmp/latest_weights.pt")
print(f"  Макс. шагов: 50000")
print("="*60 + "\n")

stats = trainer.train(max_steps=50000, device=device, text_file=train_file, block_size=512, auto_stop=True)
print(f"\nОбучение завершено: {stats}")

In [ ]:
# 4. Тест генерации
prompts = [
    "История это наука которая изучает",
    "Математика помогает человечеству",
    "Природа Земли удивительна потому что",
    "Компьютеры обрабатывают данные с помощью",
    "Человек отличается от животных тем что",
]
layer.eval()
for prompt in prompts:
    enc = tokenizer.encode(prompt)
    ids = enc.ids if hasattr(enc, 'ids') else enc
    inp = torch.tensor([ids], dtype=torch.long)
    if device == 'cuda':
        inp = inp.cuda()
    out = layer.generate(inp, max_new_tokens=80, temperature=0.7, top_p=0.9)
    print(f"\nQ: {prompt}")
    print(f"A: {tokenizer.decode(out[0].tolist())}")

In [ ]:
# 5. Упаковка для скачивания
import os, shutil

dl = '/home/jupyter/eva_download'
if os.path.exists(dl):
    shutil.rmtree(dl)
os.makedirs(dl)

if os.path.exists('/tmp/latest_weights.pt'):
    shutil.copy('/tmp/latest_weights.pt', os.path.join(dl, 'weights.pt'))
    print(f"✅ Веса: {os.path.getsize('/tmp/latest_weights.pt') / 1024 / 1024:.1f} MB")

snapshot_dir = '/home/jupyter/EVA/snapshots'
steps = sorted([d for d in os.listdir(snapshot_dir) if d.startswith('step_')]) if os.path.exists(snapshot_dir) else []
if steps:
    latest = os.path.join(snapshot_dir, steps[-1])
    shutil.copytree(latest, os.path.join(dl, 'snapshots'))
    print(f"✅ Снапшоты из: {steps[-1]}")

!cd /home/jupyter && tar czf eva_model.tar.gz eva_download/

if os.path.exists('/home/jupyter/eva_model.tar.gz'):
    size = os.path.getsize('/home/jupyter/eva_model.tar.gz') / 1024 / 1024
    print(f"\n✅ Готово: /home/jupyter/eva_model.tar.gz ({size:.1f} MB)")
    print("Скачайте: правый клик → Download")

In [ ]:
# 6. Resume
import sys, torch, pickle, faiss, os, json, time, gc
sys.path.insert(0, '/home/jupyter/EVA')
from eva.config import FCFConfig
from eva.primordial_layer import PrimordialLayer
from eva.tokenizer_utils import load_tokenizer
from eva.language_trainer import LanguageTrainer
from eva.unified_grammar import UnifiedStateGrammar

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Устройство: {device}")

config = FCFConfig()
config.training.max_steps = 50000
layer = PrimordialLayer(config)

if os.path.exists('/tmp/latest_weights.pt'):
    layer.load_state_dict(torch.load('/tmp/latest_weights.pt', map_location='cpu'))
    print("✅ Веса загружены")
if device == 'cuda':
    layer = layer.cuda()

tokenizer = load_tokenizer('/home/jupyter/EVA/tokenizer.json')
grammar = UnifiedStateGrammar(config.d_model)
trainer = LanguageTrainer(
    layer=layer, tokenizer=tokenizer, config=config,
    checkpoint_dir='/home/jupyter/EVA/checkpoints',
    state_grammar=grammar, benchmark_interval=2000,
)
trainer.checkpoint_interval = 2000

snapshot_dir = '/home/jupyter/EVA/snapshots'
os.makedirs(snapshot_dir, exist_ok=True)
steps = sorted([d for d in os.listdir(snapshot_dir) if d.startswith('step_')]) if os.path.exists(snapshot_dir) else []
if steps:
    latest = os.path.join(snapshot_dir, steps[-1])
    with open(os.path.join(latest, 'snapshots.pkl'), 'rb') as f:
        layer.state_storage.snapshots_meta = pickle.load(f)
    layer.state_storage.rebuild_from_meta()
    idx = os.path.join(latest, 'index.faiss')
    if os.path.exists(idx):
        layer.state_storage.index = faiss.read_index(idx)
    with open(os.path.join(latest, 'meta.pkl'), 'rb') as f:
        m = pickle.load(f)
        layer.meta.usage_count = m.get('usage_count', 0)
        layer.meta.confidence_history = m.get('confidence_history', [])
    print(f"✅ Снапшоты: {len(layer.state_storage.snapshots_meta)} шт.")

def save(final=False):
    p = os.path.join(snapshot_dir, f"step_{trainer.step:06d}" if not final else "final")
    os.makedirs(p, exist_ok=True)
    with open(os.path.join(p, 'snapshots.pkl'), 'wb') as f:
        pickle.dump(trainer.layer.state_storage.snapshots_meta, f)
    if trainer.layer.state_storage.index is not None:
        faiss.write_index(trainer.layer.state_storage.index, os.path.join(p, 'index.faiss'))
    with open(os.path.join(p, 'meta.pkl'), 'wb') as f:
        pickle.dump({'usage_count': trainer.layer.meta.usage_count, 'confidence_history': trainer.layer.meta.confidence_history, 'created_at': trainer.layer.meta.created_at}, f)
    trainer.layer.config.to_json(os.path.join(p, 'config.json'))
    torch.save(trainer.layer.state_dict(), '/tmp/latest_weights.pt')
    s = {'step': trainer.step, 'snapshots': len(trainer.layer.state_storage.snapshots_meta), 'confidence': trainer.layer.meta.average_confidence(), 'timestamp': time.time()}
    with open(os.path.join(p, 'status.json'), 'w') as f:
        json.dump(s, f)
    print(f"[Save] step={trainer.step} snap={s['snapshots']} conf={s['confidence']:.3f}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

trainer._save_checkpoint = save

print("\n" + "="*60)
print("  EVA — Продолжение обучения")
print("="*60)

train_file = '/home/jupyter/EVA/real_data/combined_ru.txt'
stats = trainer.train(max_steps=50000, device=device, text_file=train_file, block_size=512, auto_stop=False)
print(f"\nОбучение завершено: {stats}")